# Busy Buffet — Phase 1 Data Preparation

> **Presented by** Yanisa Mahuppon

---
## Step 0 — Import Libraries

นำเข้า library ที่จำเป็นทั้งหมดก่อนเริ่มทำงาน

In [359]:
import pandas as pd
import numpy as np
import re
from datetime import timedelta

ตั้งค่า pandas ให้แสดงผล DataFrame ได้ครบทุก column

In [360]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 120)

print('Ready')

Ready


---
## Step 1 — Load Dataset and Combine Multiple Sheets

นำเข้าข้อมูล ซึ่งข้อมูลถูกเก็บแยกรายวัน (แต่ละ sheet = 1 วัน) เราต้องรวมทั้งหมดเพื่อวิเคราะห์ภาพรวมทั้ง 5 วัน

ข้อมูลถูกเก็บแยกรายวัน (1 sheet = 1 วัน) ต้องรวมทั้งหมดเพื่อวิเคราะห์ภาพรวม


In [361]:
FILE_PATH = "../data/2026 Data Test1 Final - Busy Buffet Dataset.xlsx"

SHEET_DATE_MAP = {
    '133': '2026-03-13',
    '143': '2026-03-14',
    '153': '2026-03-15',
    '173': '2026-03-17',
    '183': '2026-03-18',
}

กำหนด column หลัก 8 ตัวที่ต้องการ (ตัด extra columns ของ sheet 183 ทิ้ง)

In [362]:
CORE_COLUMNS = [
    'service_no.', 'pax',
    'queue_start',  'queue_end',
    'table_no.',
    'meal_start',   'meal_end',
    'Guest_type'
]

วนลูปอ่านทีละ sheet แล้วใส่ `date` ก่อน concat รวมกัน

In [363]:
dfs = []

for sheet_name, date_str in SHEET_DATE_MAP.items():
    df_sheet = pd.read_excel(FILE_PATH, sheet_name=sheet_name)
    df_sheet = df_sheet[CORE_COLUMNS].copy()
    df_sheet['sheet_name'] = sheet_name
    df_sheet['date'] = pd.to_datetime(date_str)
    dfs.append(df_sheet)

raw = pd.concat(dfs, ignore_index=True)
print(f'รวมสำเร็จ — {len(raw):,} แถว, {len(raw.columns)} columns')

รวมสำเร็จ — 364 แถว, 10 columns


ตรวจสอบว่าแต่ละวันมีกี่ groups

In [364]:
raw.groupby('date')['service_no.'].count().rename('จำนวน groups')

date
2026-03-13    57
2026-03-14    81
2026-03-15    86
2026-03-17    70
2026-03-18    70
Name: จำนวน groups, dtype: int64

ดูตัวอย่างข้อมูลดิบก่อนทำความสะอาด

In [365]:
raw.head()

,service_no.,pax,queue_start,queue_end,table_no.,meal_start,meal_end,Guest_type,sheet_name,date
0,1,1.0,NaN,NaN,6,06:42:00,07:00:00,In house,133,2026-03-13
1,2,2.0,NaN,NaN,2,06:44:00,07:10:00,Walk in,133,2026-03-13
2,3,2.0,NaN,NaN,7A,07:05:00,08:08:00,Walk in,133,2026-03-13
3,4,1.0,NaN,NaN,8A,07:20:00,08:18:00,In house,133,2026-03-13
4,5,1.0,NaN,NaN,10A,07:25:00,07:45:00,In house,133,2026-03-13


---
## Step 2 — Data Quality Check

ตรวจสอบข้อมูลเบื้องต้น ก่อนทำความสะอาดข้อมูล

**สิ่งที่ตรวจสอบ:**
1. Missing values — column ไหนมีค่าว่างบ้าง
2. pax = 0 — แถวที่จำนวนคนเป็น 0 (น่าจะเป็น data entry error)
3. Guest_type — มี typo หรือค่าแปลกๆ ไหม

In [366]:
missing     = raw.isnull().sum()
missing_pct = (missing / len(raw) * 100).round(1)

pd.DataFrame({'จำนวน null': missing, '% null': missing_pct})

,จำนวน null,% null
service_no.,0,0.0
pax,1,0.3
queue_start,291,79.9
queue_end,291,79.9
table_no.,15,4.1
meal_start,15,4.1
meal_end,15,4.1
Guest_type,0,0.0
sheet_name,0,0.0
date,0,0.0


In [367]:
raw['Guest_type'].value_counts()

Guest_type
Walk in     207
In house    157
Name: count, dtype: int64

In [368]:
zero_pax = raw[raw['pax'] == 0]
print(f'พบ {len(zero_pax)} แถวที่ pax = 0')

พบ 7 แถวที่ pax = 0


In [369]:
cols_show = ['date', 'service_no.', 'pax', 'queue_start', 'meal_start', 'Guest_type']
zero_pax[cols_show]

,date,service_no.,pax,queue_start,meal_start,Guest_type
144,2026-03-15,7,0.0,NaN,07:15:00,Walk in
158,2026-03-15,21,0.0,NaN,08:12:00,In house
165,2026-03-15,28,0.0,08:16:00,NaN,In house
177,2026-03-15,40,0.0,08:50:00,NaN,In house
186,2026-03-15,49,0.0,09:10:00,NaN,Walk in
197,2026-03-15,60,0.0,09:40:00,NaN,Walk in
215,2026-03-15,78,0.0,11:14:00,NaN,Walk in


---
## Step 3 — Data Cleaning

แก้ปัญหาที่ตรวจเจอใน Step 2 ทีละอย่าง

1. **Standardize Guest_type** — trim whitespace ป้องกัน typo แฝง
2. **Flag pax = 0** — เก็บไว้ก่อน เพราะยังมีข้อมูลเวลาที่มีประโยชน์ (เช่น walk-away ที่ลืมบันทึก pax)
3. **แปลง time columns** — จาก `datetime.time` → `timedelta` เพื่อให้ลบกันได้
4. **ตัดแถวไร้ข้อมูล** — แถวที่ไม่มีทั้ง queue และ meal วิเคราะห์ไม่ได้

In [370]:
df = raw.copy()

In [371]:
# Standardize Guest_type
df['Guest_type'] = df['Guest_type'].str.strip()

assert set(df['Guest_type'].dropna().unique()) <= {'In house', 'Walk in'}
print('Guest_type สะอาดแล้ว')

Guest_type สะอาดแล้ว


In [372]:
# Flag แถวที่ pax = 0 (ไม่ตัดทิ้ง)
df['has_pax_issue'] = df['pax'] == 0

print(f'แถวที่มี pax = 0: {df["has_pax_issue"].sum()} แถว')

แถวที่มี pax = 0: 7 แถว


สร้างฟังก์ชันแปลง `datetime.time` → `timedelta`  
เพราะ pandas อ่านเวลาจาก Excel มาเป็น `datetime.time` object ซึ่งลบกันตรงๆ ไม่ได้

In [373]:
def to_timedelta(val):
    if pd.isna(val):
        return pd.NaT
    if hasattr(val, 'hour'):
        return timedelta(hours=val.hour, minutes=val.minute)
    return pd.NaT

In [374]:
time_cols = ['queue_start', 'queue_end', 'meal_start', 'meal_end']

for col in time_cols:
    df[col] = df[col].apply(to_timedelta)

print('แปลง time columns สำเร็จ')
print(df[time_cols].dtypes)

แปลง time columns สำเร็จ
queue_start    timedelta64[us]
queue_end      timedelta64[us]
meal_start     timedelta64[us]
meal_end       timedelta64[us]
dtype: object


In [375]:
no_data_mask = df['queue_start'].isna() & df['meal_start'].isna()

print(f'แถวที่จะถูกตัด : {no_data_mask.sum()} แถว')
df = df[~no_data_mask].copy()
print(f'แถวที่เหลือ    : {len(df):,} แถว')

แถวที่จะถูกตัด : 1 แถว
แถวที่เหลือ    : 363 แถว


---
## Step 4 — Create Derived Columns 

สร้าง column ใหม่จากข้อมูลที่มี เพื่อให้ตอบโจทย์ Task 1 และ Task 2

| Column ใหม่ | นิยาม | ใช้ใน Task |
|-------------|-------|------------|
| `is_queued` | มี queue_start หรือเปล่า | Task 1 Comment 1 |
| `is_walkaway` | รอคิวแต่ไม่ได้นั่ง | Task 1 Comment 1 |
| `wait_time_min` | เวลารอคิว (นาที) | Task 1 Comment 1, Task 2 Action 3 |
| `meal_duration_min` | ระยะเวลานั่งกิน (นาที) | Task 1 Comment 3, Task 2 Action 1 |
| `day_of_week` | วันในสัปดาห์ | Task 2 Action 2 |
| `is_weekend` | เสาร์-อาทิตย์หรือเปล่า | Task 2 Action 2 |

**Logic จาก Appendix 1:**
```
Walk-away     = queue_start มีค่า AND meal_start ไม่มีค่า
wait_time     = queue_end − queue_start
meal_duration = meal_end  − meal_start
```

In [376]:
df['is_queued']   = df['queue_start'].notna()
df['is_walkaway'] = df['queue_start'].notna() & df['meal_start'].isna()
df['is_direct']   = df['queue_start'].isna()  & df['meal_start'].notna()

print('is_queued   :', df['is_queued'].sum())
print('is_walkaway :', df['is_walkaway'].sum())
print('is_direct   :', df['is_direct'].sum())

is_queued   : 73
is_walkaway : 14
is_direct   : 290


คำนวณ wait time — ใช้ `np.where` เพื่อให้คำนวณเฉพาะแถวที่มีข้อมูลครบ

In [377]:
# Wait time (นาที)
has_queue = df['queue_start'].notna() & df['queue_end'].notna()

df['wait_time_min'] = np.where(
    has_queue,
    (df['queue_end'] - df['queue_start']).dt.total_seconds() / 60,
    np.nan
)

df['wait_time_min'] = df['wait_time_min'].clip(lower=0)

In [378]:
# Meal duration (นาที)
has_meal = df['meal_start'].notna() & df['meal_end'].notna()

df['meal_duration_min'] = np.where(
    has_meal,
    (df['meal_end'] - df['meal_start']).dt.total_seconds() / 60,
    np.nan
)

df['meal_duration_min'] = df['meal_duration_min'].clip(lower=0)

In [379]:
# Day of week และ weekend flag
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend']  = df['date'].dt.dayofweek >= 5

ดูค่าเฉลี่ยเบื้องต้นเพื่อ sanity check

In [380]:
# Sanity check
print('wait_time_min เฉลี่ย (เฉพาะแถวที่รอจริง):')
print(df.loc[df['wait_time_min'] > 0, 'wait_time_min'].describe().round(1))

wait_time_min เฉลี่ย (เฉพาะแถวที่รอจริง):
count    73.0
mean     34.8
std      21.1
min       1.0
25%      15.0
50%      32.0
75%      50.0
max      80.0
Name: wait_time_min, dtype: float64


In [381]:
print('meal_duration_min เฉลี่ย แยก Guest_type:')
print(df.groupby('Guest_type')['meal_duration_min'].mean().round(1))

meal_duration_min เฉลี่ย แยก Guest_type:
Guest_type
In house    45.5
Walk in     72.8
Name: meal_duration_min, dtype: float64


---
## Step 5 — Normalize table_no. and Map Zone

In [382]:
# กำหนด zone sets ตาม Appendix 2
INDOOR_TABLES = {
    '1A','1B','2A','2B',
    '3A','3B','4A','4B',
    '5A','5B','6A','6B'
}

OUTDOOR_TABLES = {
    '7A','7B','7C',
    '8A','8B','8C',
    '9A','9B','9C',
    '10A','10B',
    '11A','11B',
    '12','13','14','15'
}

In [383]:
# ฟังก์ชันแตก table_no. เป็น list ของ individual units
def parse_table_no(raw_val):
    if pd.isna(raw_val):
        return []
    s = str(raw_val).strip()
    if re.match(r'^\d+\.0$', s):  # แปลง '7.0' → '7'
        s = s[:-2]
    parts = re.split(r'[-/]', s)  # แยกด้วย - หรือ /
    return [p.strip() for p in parts if p.strip()]

In [384]:
# ฟังก์ชันหา zone ของโต๊ะหน่วยเดียว
def get_zone(table_unit):
    t = str(table_unit).upper().strip()
    if t == '99':           return 'Queue Area'
    if t in INDOOR_TABLES:  return 'Indoor'
    if t in OUTDOOR_TABLES: return 'Outdoor'
    return 'Unknown'

In [385]:
# ฟังก์ชันสรุป zone ของ group (อาจนั่งหลายโต๊ะ)
def summarize_zone(zones):
    unique = list(set(zones))
    if len(unique) == 1:
        return unique[0]
    return 'Mixed'

In [386]:
# Apply ฟังก์ชันทั้งหมดเพื่อสร้าง columns ใหม่
df['table_units'] = df['table_no.'].apply(parse_table_no)
df['n_tables']    = df['table_units'].apply(len)
df['is_merged']   = df['n_tables'] > 1

In [387]:
df['table_zones'] = df['table_units'].apply(
    lambda units: [get_zone(u) for u in units] if units else ['Unknown']
)

df['zone'] = df['table_zones'].apply(summarize_zone)

In [388]:
# ดูการกระจาย zone
print('Parse table_no. สำเร็จ\n')
print(df['zone'].value_counts())
print(f'\nกลุ่มที่ใช้โต๊ะ merged: {df["is_merged"].sum()} กลุ่ม')

Parse table_no. สำเร็จ

zone
Outdoor       165
Unknown       122
Indoor         74
Queue Area      1
Mixed           1
Name: count, dtype: int64

กลุ่มที่ใช้โต๊ะ merged: 15 กลุ่ม


In [389]:
# ดูโต๊ะที่ zone = Unknown เพื่อตรวจสอบ
cols_show = ['date', 'service_no.', 'table_no.', 'table_units', 'zone']

df[df['zone'] == 'Unknown'][cols_show]

,date,service_no.,table_no.,table_units,zone
0,2026-03-13,1,6,[6],Unknown
1,2026-03-13,2,2,[2],Unknown
6,2026-03-13,7,9,[9],Unknown
7,2026-03-13,8,4,[4],Unknown
8,2026-03-13,9,11,[11],Unknown
...,...,...,...,...,...
336,2026-03-18,43,16,[16],Unknown
344,2026-03-18,51,9,[9],Unknown
350,2026-03-18,57,3,[3],Unknown
353,2026-03-18,60,16,[16],Unknown


---
## Step 6 — Create Seating Status Column

```
Direct     → นั่งได้ทันทีไม่ต้องรอ
Queued     → รอคิวแล้วได้นั่ง
Walk-away  → รอคิวแต่ออกไปโดยไม่ได้นั่ง
```

In [390]:
# ฟังก์ชัน classify
def classify_status(row):
    if row['is_walkaway']:
        return 'Walk-away'
    if row['is_queued']:
        return 'Queued'
    return 'Direct'

In [391]:
# Apply
df['seating_status'] = df.apply(classify_status, axis=1)

print('seating_status สร้างสำเร็จ')

seating_status สร้างสำเร็จ


In [392]:
# ดูสรุปรายวัน
df.groupby(['date', 'seating_status']).size().unstack(fill_value=0)

seating_status,Direct,Queued,Walk-away
date,,,
2026-03-13,57,0,0
2026-03-14,62,18,1
2026-03-15,32,41,13
2026-03-17,69,0,0
2026-03-18,70,0,0


In [393]:
# ดูแยก Guest_type
df.groupby(['Guest_type', 'seating_status']).size().unstack(fill_value=0)

seating_status,Direct,Queued,Walk-away
Guest_type,,,
In house,132,18,7
Walk in,158,41,7


---
## Step 7 — Final QA Check

ตรวจสอบ DataFrame สุดท้ายก่อน export
- Shape และ data types
- Outlier ใน wait_time และ meal_duration
- ดูตัวอย่างข้อมูล

In [394]:
# ดู shape และ null count ทุก column
pd.DataFrame({
    'dtype':      df.dtypes,
    'null_count': df.isnull().sum()
})

,dtype,null_count
service_no.,int64,0
pax,float64,0
queue_start,timedelta64[us],290
queue_end,timedelta64[us],290
table_no.,object,14
meal_start,timedelta64[us],14
meal_end,timedelta64[us],14
Guest_type,str,0
sheet_name,str,0
date,datetime64[us],0


In [395]:
# Top 5 wait time นานสุด
cols = ['date', 'Guest_type', 'seating_status', 'wait_time_min']

df.nlargest(5, 'wait_time_min')[cols]

,date,Guest_type,seating_status,wait_time_min
205,2026-03-15,Walk in,Queued,80.0
201,2026-03-15,Walk in,Queued,76.0
200,2026-03-15,In house,Queued,69.0
194,2026-03-15,Walk in,Queued,68.0
197,2026-03-15,Walk in,Walk-away,68.0


In [396]:
# Top 5 meal duration นานสุด
cols = ['date', 'Guest_type', 'seating_status', 'meal_duration_min']

df.nlargest(5, 'meal_duration_min')[cols]

,date,Guest_type,seating_status,meal_duration_min
298,2026-03-18,In house,Direct,321.0
57,2026-03-14,Walk in,Direct,225.0
242,2026-03-17,Walk in,Direct,220.0
269,2026-03-17,In house,Direct,217.0
158,2026-03-15,In house,Direct,173.0


In [397]:
# ดูตัวอย่าง DataFrame สุดท้าย
preview_cols = [
    'date', 'day_of_week', 'service_no.', 'pax',
    'Guest_type', 'seating_status',
    'wait_time_min', 'meal_duration_min',
    'table_no.', 'zone', 'is_weekend'
]

df[preview_cols].head(10)

,date,day_of_week,service_no.,pax,Guest_type,seating_status,wait_time_min,meal_duration_min,table_no.,zone,is_weekend
0,2026-03-13,Friday,1,1.0,In house,Direct,NaN,18.0,6,Unknown,False
1,2026-03-13,Friday,2,2.0,Walk in,Direct,NaN,26.0,2,Unknown,False
2,2026-03-13,Friday,3,2.0,Walk in,Direct,NaN,63.0,7A,Outdoor,False
3,2026-03-13,Friday,4,1.0,In house,Direct,NaN,58.0,8A,Outdoor,False
4,2026-03-13,Friday,5,1.0,In house,Direct,NaN,20.0,10A,Outdoor,False
5,2026-03-13,Friday,6,1.0,In house,Direct,NaN,52.0,8B,Outdoor,False
6,2026-03-13,Friday,7,1.0,In house,Direct,NaN,32.0,9,Unknown,False
7,2026-03-13,Friday,8,5.0,Walk in,Direct,NaN,58.0,4,Unknown,False
8,2026-03-13,Friday,9,4.0,In house,Direct,NaN,33.0,11,Unknown,False
9,2026-03-13,Friday,10,2.0,In house,Direct,NaN,47.0,16,Unknown,False


---
## Step 8 — Export DataFrame For Phase 2

In [398]:
# กำหนด output paths
OUTPUT_CSV    = '../data/busy_buffet_clean.csv'
OUTPUT_PICKLE = '../data/busy_buffet_clean.pkl'

In [399]:
# Export
df.to_csv(OUTPUT_CSV, index=False)
df.to_pickle(OUTPUT_PICKLE)

print(f'Export สำเร็จ')
print(f'   CSV    → {OUTPUT_CSV}')
print(f'   Pickle → {OUTPUT_PICKLE}')

Export สำเร็จ
   CSV    → ../data/busy_buffet_clean.csv
   Pickle → ../data/busy_buffet_clean.pkl


In [400]:
# แสดง columns ทั้งหมดที่ส่งต่อไป Phase 2
for col in df.columns:
    print(f'  - {col}')

  - service_no.
  - pax
  - queue_start
  - queue_end
  - table_no.
  - meal_start
  - meal_end
  - Guest_type
  - sheet_name
  - date
  - has_pax_issue
  - is_queued
  - is_walkaway
  - is_direct
  - wait_time_min
  - meal_duration_min
  - day_of_week
  - is_weekend
  - table_units
  - n_tables
  - is_merged
  - table_zones
  - zone
  - seating_status


---
## สรุป Phase 1 — Data Preparation

| Step | สิ่งที่ทำ | Output |
|------|-----------|--------|
| 1 | รวม 5 sheets (133–183) + เพิ่ม `date` และ `sheet_name` / ตัด extra columns ของ sheet 183 | DataFrame ดิบ 364 แถว |
| 2 | Data Quality Check — ตรวจ null, `Guest_type` typo, แถวที่ `pax = 0` | ระบุปัญหาทั้งหมด |
| 3 | Clean: standardize `Guest_type`, flag `pax=0` (ไม่ตัดทิ้ง), แปลง time → timedelta, ตัดแถวที่ไม่มีข้อมูลเวลาเลย | DataFrame สะอาด |
| 4 | Derived columns: `is_queued`, `is_walkaway`, `is_direct`, `wait_time_min`, `meal_duration_min`, `day_of_week`, `is_weekend` | 7 columns ใหม่ |
| 5 | Parse `table_no.` (รองรับ `-` และ `/`) → zone mapping ตาม Appendix 2 (Indoor / Outdoor / Queue Area) | `zone`, `is_merged`, `n_tables` |
| 6 | Seating status classifier: Direct / Queued / Walk-away | `seating_status` |
| 7 | Final QA — ตรวจ shape, null count, top-5 outlier ของ wait time และ meal duration | ผ่านทุกจุด |
| 8 | Export 2 format: CSV (สำหรับ Streamlit) + Pickle (preserve dtypes สำหรับ Phase 2) | `busy_buffet_clean.csv` + `.pkl` |